# Cleaning Validation & Data Quality Audit — CatalogIQ

**Version 3.** Second review round. v2 fixed how the checks were scoped; this
round fixes what two of them actually measured. Two v2 findings were wrong and
are withdrawn below.

| What this review round found | What changed here |
|---|---|
| The category comparison removed *any* repeated path element | §7 allows a collapse only when the **whole path** repeats exactly |
| The dtype finding compared pandas inference, not the files | §4 reads both files as text and compares stored values |
| Disappearing labels were excused by a whitespace rule | §6 traces every one to its **raw rows** and requires a verified rule or an approved relabel |
| Quarantine reasons were only checked for presence | §2.1 re-derives each reason's documented conditions from the original row |
| Isolated bad values were treated like policy breaches | §8–§10 report them as **diagnostics**; FAIL is reserved for a rule actually broken |

**Two v2 results are withdrawn.**

- v2 reported `ProductCategory` explained by "spacing and duplicate collapse",
  but the test collapsed any repeated element, so `A>B>A>C` → `A>B>C` would have
  passed. §7 now uses the strict rule. The answer does not change; the check is
  now sound rather than accidentally right.
- v2 reported that "111,249 values change representation in the export". §4 shows
  that figure was produced by pandas inferring `float64`, not by either file. As
  stored, the candidate and the export are **identical** in those columns.

**Standing rules.** A PASS must come from a check that ran over the full
population it claims. A FAIL means the pipeline's action broke a documented rule
— not that a questionable value survived somewhere.

## 0. Setup — one explicit run

The run directory is chosen explicitly. Auto-detection across `data/processed/`
can pair a candidate file from one run with a decision table from another, and
the resulting numbers would reconcile against nothing.

Set `RUN_NAME` to the directory under `data/processed/`. Leave it `None` to use
the newest, which is printed and must be confirmed against the one being
audited.

In [ ]:
import pandas as pd, re, json
from pathlib import Path

pd.set_option("display.max_rows", 250); pd.set_option("display.width", 170)

# The run directory under data/processed/ to audit. Set it: leaving it None lets
# the notebook pick the newest, which is how artifacts from two runs get mixed.
RUN_NAME = "my_cleaning_run"

CATALOGIQ = None
try:
    import sys
    for _p in (Path.cwd(), *Path.cwd().parents):
        if (_p / "src" / "catalogiq" / "paths.py").is_file():
            sys.path.insert(0, str(_p / "src")); break
    from catalogiq import paths as CATALOGIQ
except Exception:
    pass

def project_root(start="."):
    if CATALOGIQ is not None: return CATALOGIQ.ROOT_PATH
    p = Path(start).resolve()
    for _ in range(8):
        if (p/"data"/"provided").is_dir() or (p/".git").exists(): return p
        if p.parent == p: break
        p = p.parent
    return Path.cwd()

ROOT = project_root()
DATA = CATALOGIQ.DATA_PATH if CATALOGIQ is not None else ROOT/"data"
PROVIDED, PROCESSED = DATA/"provided", DATA/"processed"
print("project root:", ROOT)

runs = sorted([d for d in PROCESSED.glob("*") if d.is_dir() and (d/"summary.json").is_file()],
              key=lambda d: d.stat().st_mtime, reverse=True) if PROCESSED.is_dir() else []
if not runs:
    raise SystemExit(f"No run directory with a summary.json under {PROCESSED}. "
                     f"Run: python -m catalogiq --mode integrated --output-dir data/processed/<name>")
RUN = (PROCESSED/RUN_NAME) if RUN_NAME else runs[0]
if not (RUN/"summary.json").is_file():
    print(f"RUN_NAME is set to {RUN_NAME!r} but {RUN} has no summary.json.")
    print(f"Available runs: {[d.name for d in runs]}")
    raise SystemExit("Set RUN_NAME to one of the runs listed above.")

print("runs available :", [d.name for d in runs])
print("AUDITING RUN   :", RUN.name)
if RUN_NAME is None:
    print("  ! RUN_NAME is None, so the newest run was chosen automatically.")
    print("  ! Set it explicitly - this is how files from two runs get mixed.")

SUMMARY = json.loads((RUN/"summary.json").read_text())
print("policy version :", SUMMARY.get("policy_version", "unknown"))

TARGETS = ["Mnfr","Brand","Platform","Segment","Sub-Segment","TargetAgeGroup"]
NUMERIC = ["ProductRating","ProductReviewsCount","XRatXRev","ReviewsCount"]
URLCOLS = ["ProductUrl","ProductImageUrl"]

DATASETS = {}
for tag, rawname, candname in [
        ("training","Selfcare_Training_data (1).csv","training_candidate.csv"),
        ("target",  "Selfcare_Target_data (1).csv",  "target_candidate.csv")]:
    rp, cp = PROVIDED/rawname, RUN/candname
    if not rp.is_file(): print(f"  {tag}: raw missing at {rp}"); continue
    if not cp.is_file(): print(f"  {tag}: candidate missing at {cp}"); continue
    raw  = pd.read_csv(rp, dtype="string", keep_default_na=False, low_memory=False)
    cand = pd.read_csv(cp, dtype="string", keep_default_na=False, low_memory=False)
    raw["source_row"] = (raw.index + 1).astype(str)   # summary.json: source_row_base = 1
    DATASETS[tag] = {"raw": raw, "cand": cand, "run": RUN}
    print(f"  {tag:9s} raw {len(raw):>7,}  candidate {len(cand):>7,}")

if not DATASETS:
    raise SystemExit("No dataset could be loaded. Put the PO's CSVs in data/provided/.")

RD = pd.read_csv(RUN/"row_decisions.csv", dtype=str, low_memory=False)
LABEL_CHANGES = (pd.read_csv(RUN/"label_changes.csv", dtype=str)
                 if (RUN/"label_changes.csv").is_file() else pd.DataFrame())
print(f"\nrow_decisions {len(RD):,} rows   label_changes {len(LABEL_CHANGES):,} rows")

### 0.1 Verdict recorder, and what each status means

A verdict is about **what the pipeline did**. A diagnostic is about **a value
that is questionable but that no rule forbids**. v2 blurred the two, which is
how an isolated bad URL ended up carrying the same weight as a broken policy.

| Status | Means |
|---|---|
| PASS | The action complies with a rule in `docs/decisions.md` or `docs/structural_check.md`, verified over the stated population |
| WARN | The action complies, but it is not recorded in `decisions.md`, or the reason code it recorded is not satisfied by that code's own documented wording |
| FAIL | The action **violates** a documented rule |
| BLOCKED | There is nothing to compare, so no verdict is possible |
| *diagnostic* | A questionable value or a cost worth reporting. Not a verdict on the pipeline, and it never becomes a FAIL on its own |

A PASS whose coverage is below the population it claims is downgraded
automatically, which is the guard v1 needed and did not have.

In [ ]:
VERDICTS, DIAGNOSTICS = [], []

def verdict(dataset, criterion, status, detail, covered=None, population=None):
    if status == "PASS" and covered is not None and population and covered < population:
        status = "WARN"
        detail = (f"Check covered {covered:,} of {population:,} rows, so this "
                  f"cannot be reported as PASS. " + detail)
    VERDICTS.append({"dataset":dataset,"criterion":criterion,"status":status,
                     "covered":covered,"population":population,"detail":detail})
    bar = {"PASS":"","WARN":"!  ","FAIL":"!! ","BLOCKED":"-- "}[status]
    cov = f"  [{covered:,}/{population:,}]" if covered is not None and population else ""
    print(f"\n[{dataset}] {criterion}: {bar}{status}{cov}")
    for line in str(detail).split("\n"): print("   " + line)

def diagnostic(dataset, topic, detail):
    """A value worth reporting that no documented rule prohibits."""
    DIAGNOSTICS.append({"dataset":dataset,"topic":topic,"detail":detail})
    print(f"\n[{dataset}] diagnostic - {topic}")
    for line in str(detail).split("\n"): print("   " + line)

def blank(s):
    if isinstance(s, pd.DataFrame):
        return s.apply(lambda col: col.str.strip().str.lower().isin(["", "null"]))
    return s.str.strip().str.lower().isin(["", "null"])
def live(s):   return s[~blank(s)]

## 1. Row counts, and the same rows

Counts reconciling is necessary and not sufficient. Two disjoint sets of 614
rows also reconcile. This compares the **sets**: the rows absent from the
candidate file must be exactly the rows the quarantine partition and the
decision table name.

In [ ]:
for tag, D in DATASETS.items():
    raw, cand = D["raw"], D["cand"]
    removed = set(raw["source_row"]) - set(cand["source_row"])
    added   = set(cand["source_row"]) - set(raw["source_row"])
    rd = RD[RD["dataset"] == tag]
    tab = set(rd[rd["quarantine_reasons"].notna()]["source_row"])
    qf  = RUN / f"{tag}_quarantine.csv"
    filep = set(pd.read_csv(qf, dtype=str)["source_row"]) if qf.is_file() else None

    print(f"--- {tag}: raw {len(raw):,} -> candidate {len(cand):,}  removed {len(removed):,}")
    print(f"    decision table names {len(tab):,} quarantined")
    if filep is not None: print(f"    {qf.name} holds {len(filep):,}")
    print(f"    every raw row has a decision row: {set(raw['source_row']) <= set(rd['source_row'])}")

    problems = []
    if added: problems.append(f"{len(added):,} rows in candidate that are not in raw")
    if removed != tab:
        problems.append(f"removed set != decision table: {len(removed ^ tab):,} rows differ")
    if filep is not None and removed != filep:
        problems.append(f"removed set != {qf.name}: {len(removed ^ filep):,} rows differ")
    if not set(raw["source_row"]) <= set(rd["source_row"]):
        problems.append("some raw rows have no row in the decision table")

    D["removed"] = removed
    if problems:
        verdict(tag, "row counts and identity", "FAIL", "\n".join("  "+p for p in problems),
                len(raw), len(raw))
    else:
        verdict(tag, "row counts and identity", "PASS",
                f"{len(removed):,} rows removed ({len(removed)/len(raw):.2%}), and they are "
                f"exactly the rows named by the decision table"
                + (" and the quarantine partition" if filep is not None else "")
                + ". Symmetric difference is zero, not merely equal totals.",
                len(raw), len(raw))

## 2. Quarantine reasons

Every removed row must name the rule that removed it, and the reason must be
present in the row's own content. §9 checks the content; this checks coverage.

In [ ]:
for tag, D in DATASETS.items():
    rd = RD[RD["dataset"] == tag]
    q  = rd[rd["quarantine_reasons"].notna()]
    removed = D["removed"]
    print(f"--- {tag}: {len(q):,} rows carry a reason, {len(removed):,} were removed")
    print(q["quarantine_reasons"].value_counts().head(8).to_string())
    missing = removed - set(q["source_row"])
    holds = rd["hold_reasons"].notna().sum()
    print(f"\n    rows held (hold_reasons): {holds}")
    if missing:
        verdict(tag, "quarantine reasons", "FAIL",
                f"{len(missing):,} removed rows carry no reason.", len(removed), len(removed))
    else:
        verdict(tag, "quarantine reasons", "PASS",
                f"All {len(removed):,} removed rows name a rule. hold_reasons empty on "
                f"all {len(rd):,} rows, matching decision 6 (no automatic whole-row hold).",
                len(removed), len(removed))

## 2.1 Every quarantine reason re-derived from the original row

§2 only checks that a reason is *present*. That is not an audit: the pipeline
could record any reason it liked. This section takes each recorded reason code
and independently re-evaluates its **documented** conditions against the raw
row, using `docs/structural_check.md`:

| Reason code | Required evidence, as documented |
|---|---|
| `missing_product_text` | ProductName, ProductDescription **and** ProductContents all missing |
| `text_in_multiple_numeric_anchors` | Nonnumeric text in **at least two** measurement fields |
| `text_numeric_failure_with_corroboration` | Nonnumeric text in one measurement **plus** another invalid measurement, a displaced identifier, or an invalid URL field |
| `url_in_timestamp_with_displacement` | HTTP(S) URL in the timestamp **plus** numeric text or another displaced identifier / invalid URL. Timestamp alone is nothing |
| `unexpected_exclude_with_populated_unknown_columns` | Non-marker `Exclude` content **together with** populated `Sun1`–`Sun5`/`Notes` |
| `multiple_suspicious_targets` | **At least two** targets hold a URL or fail the working vocabulary |

Missing is the documented definition: trimmed blank or case-insensitive `null`.
Anchors are `ProductRating`, `ProductReviewsCount`, `ReviewsCount`, `XRatXRev`;
measurements must be finite and nonnegative, and count fields integral.

Every row whose reason is not supported is printed. A row is only a wrongly
removed row if **none** of its recorded codes holds.

In [ ]:
ANCHORS = ["ProductRating","ProductReviewsCount","ReviewsCount","XRatXRev"]
COUNTCOLS = {"ProductReviewsCount","ReviewsCount"}
IDCOLS = ["Sku","Upc","ProductModelNumber","MDM_Id","JoiningKey"]
VOCAB = {"Mnfr":{"All others","J&J"}, "TargetAgeGroup":{"Adult","Children","Infant"}}
URLRE = re.compile(r"https?://", re.I)

def _missing(v):
    s = str(v).strip(); return s == "" or s.lower() == "null"

def _bad_measure(col, v):
    if _missing(v): return False          # a missing anchor is a diagnostic, not text
    try: x = float(str(v).strip())
    except Exception: return True         # nonnumeric text
    if x != x or x in (float("inf"), float("-inf")): return True
    if x < 0: return True                               # must be nonnegative
    if col in COUNTCOLS and x != int(x): return True    # counts must be integral
    return False

def _bad_url(v):   return (not _missing(v)) and not URLRE.search(str(v))
def _url_in(v):    return (not _missing(v)) and bool(URLRE.search(str(v)))

def row_signals(r):
    n_anchor = sum(_bad_measure(c, r[c]) for c in ANCHORS if c in r.index)
    n_url    = sum(_bad_url(r[c]) for c in URLCOLS if c in r.index)
    n_id     = sum(_url_in(r[c]) for c in IDCOLS if c in r.index)
    ts_url   = _url_in(r["MDM_InsertDateTime"]) if "MDM_InsertDateTime" in r.index else False
    n_tgt = 0
    for t in TARGETS:
        if t not in r.index or _missing(r[t]): continue
        v = str(r[t]).strip()
        if URLRE.search(v) or (t in VOCAB and v not in VOCAB[t]): n_tgt += 1
    sun = any(not _missing(r[c]) for c in ["Sun1","Sun2","Sun3","Sun4","Sun5","Notes"]
              if c in r.index)
    exc = ("Exclude" in r.index and not _missing(r["Exclude"])
           and str(r["Exclude"]).strip().lower() != "exclude")
    return dict(anchor=n_anchor, url=n_url, ident=n_id, ts_url=ts_url,
                tgt=n_tgt, sun=sun, exc=exc)

RULES = {
 "missing_product_text":
   lambda r,s: all(_missing(r[c]) for c in
                   ["ProductName","ProductDescription","ProductContents"] if c in r.index),
 "text_in_multiple_numeric_anchors":
   lambda r,s: s["anchor"] >= 2,
 "text_numeric_failure_with_corroboration":
   lambda r,s: s["anchor"] >= 1 and (s["anchor"] >= 2 or s["ident"] >= 1 or s["url"] >= 1),
 "url_in_timestamp_with_displacement":
   lambda r,s: s["ts_url"] and (s["anchor"] >= 1 or s["ident"] >= 1 or s["url"] >= 1),
 "unexpected_exclude_with_populated_unknown_columns":
   lambda r,s: s["exc"] and s["sun"],
 "multiple_suspicious_targets":
   lambda r,s: s["tgt"] >= 2,
}

for tag, D in DATASETS.items():
    raw = D["raw"].set_index("source_row")
    rdq = RD[(RD["dataset"] == tag) & RD["quarantine_reasons"].notna()]
    tally, unsupported, orphan_rows = {}, [], []

    for _, dec in rdq.iterrows():
        sr = dec["source_row"]
        if sr not in raw.index: continue
        r = raw.loc[sr]
        s = row_signals(r)
        codes = {c.split(":",1)[1] for c in str(dec["quarantine_reasons"]).split(";") if ":" in c}
        checked = {c: RULES[c](r, s) for c in codes if c in RULES}
        for c, ok in checked.items():
            t = tally.setdefault(c, [0,0]); t[0] += 1; t[1] += int(ok)
            if not ok: unsupported.append((sr, c, s))
        if checked and not any(checked.values()):
            orphan_rows.append((sr, sorted(codes)))

    print(f"--- {tag}: {len(rdq):,} quarantined rows re-derived from raw")
    print(f"    {'reason code':52}{'rows':>7}{'held':>7}{'not held':>10}")
    for c in sorted(tally):
        n, ok = tally[c]
        print(f"    {c:52}{n:>7,}{ok:>7,}{n-ok:>10,}")

    if unsupported:
        print(f"\n    {len(unsupported)} reason code(s) not supported by their own wording:")
        for sr, c, s in unsupported[:6]:
            print(f"      source_row {sr}: {c}")
            print(f"        signals: bad anchors {s['anchor']}, bad URLs {s['url']}, "
                  f"displaced ids {s['ident']}, URL in timestamp {s['ts_url']}, "
                  f"suspicious targets {s['tgt']}")
        if len(unsupported) > 6: print(f"      ... and {len(unsupported)-6} more")

    pop = len(rdq)
    if orphan_rows:
        verdict(tag, "quarantine conditions re-derived", "FAIL",
                f"{len(orphan_rows)} removed row(s) have no recorded reason whose documented "
                f"conditions hold on the original row. These rows were removed without "
                f"support:\n" + "\n".join(f"  source_row {sr}: {cs}" for sr, cs in orphan_rows[:10]),
                pop, pop)
    elif unsupported:
        verdict(tag, "quarantine conditions re-derived", "WARN",
                f"Every one of the {pop:,} removals is supported: each row has at least one "
                f"recorded reason whose documented conditions hold on the raw row. But "
                f"{len(unsupported)} row(s) carry a second code, "
                f"text_numeric_failure_with_corroboration, whose documented corroboration list "
                f"does not include a URL in the timestamp - while url_in_timestamp_with_"
                f"displacement, which those same rows also carry, does accept numeric text as "
                f"corroboration. The two rules corroborate each other in one direction only. "
                f"The removal stands; the reason labelling is imprecise and the wording in "
                f"docs/structural_check.md should say so.", pop, pop)
    else:
        verdict(tag, "quarantine conditions re-derived", "PASS",
                f"All {pop:,} removals verified against the documented conditions, re-derived "
                f"from the original CSV rather than taken from the pipeline's own record.",
                pop, pop)

## 3. Missingness

Cleaning must not make a column emptier. Measured over the retained rows only,
since removed rows are accounted for separately.

In [ ]:
for tag, D in DATASETS.items():
    raw, cand = D["raw"], D["cand"]
    kept = raw[raw["source_row"].isin(set(cand["source_row"]))]
    shared = [c for c in raw.columns if c in cand.columns and c != "source_row"]
    m = pd.DataFrame({"raw": blank(kept[shared]).mean()*100,
                      "cleaned": blank(cand[shared]).mean()*100})
    m["change_pp"] = m["cleaned"] - m["raw"]
    worse = m[m["change_pp"] > 0.5]
    print(f"--- {tag}: {len(shared)} shared columns over {len(kept):,} retained rows")
    print(m.sort_values("change_pp", ascending=False).head(8).round(2).to_string())
    if worse.empty:
        verdict(tag, "missingness", "PASS",
                "No shared column became more than 0.5pp emptier among retained rows.",
                len(kept), len(cand))
    else:
        verdict(tag, "missingness", "FAIL",
                "Columns that lost data:\n" +
                "\n".join(f"  {c}: +{r['change_pp']:.2f}pp" for c, r in worse.iterrows()),
                len(kept), len(cand))

## 4. Candidate against export, as the files actually store them

**This is where v2 was wrong.** v2 reported that "111,249 values change
representation in the export: `5` becomes `5.0`". That number did not come from
either file. It came from pandas reading a column as `float64` and the check
stringifying the result.

The honest comparison reads **both** files with `dtype="string"` and
`keep_default_na=False`, so no inference happens at all, joins on
`dataset` + `source_sha256` + `source_row`, and compares the stored characters.
What pandas infers is reported separately, as a diagnostic, because it is a
property of how a consumer reads the file and not of what the pipeline wrote.

In [ ]:
exp_path = PROCESSED / "training_cleaned.csv"
KEY = ["dataset","source_sha256","source_row"]

if not exp_path.is_file():
    verdict("training", "candidate vs export (stored values)", "BLOCKED",
            f"No {exp_path.name}; run scripts/export_training_dataset.py --run-dir {RUN}")
else:
    cand_s = pd.read_csv(RUN/"training_candidate.csv", dtype="string",
                         keep_default_na=False, low_memory=False)
    exp_s  = pd.read_csv(exp_path, dtype="string",
                         keep_default_na=False, low_memory=False)
    cols = [c for c in NUMERIC if c in exp_s.columns and c in cand_s.columns]
    keys = [k for k in KEY if k in exp_s.columns and k in cand_s.columns]
    print(f"joining on {keys}")

    j = exp_s[keys+cols].merge(cand_s[keys+cols], on=keys,
                               suffixes=("_e","_c"), how="inner")
    print(f"export rows {len(exp_s):,}   joined {len(j):,}")

    rows = []
    for c in cols:
        d = j[c+"_e"] != j[c+"_c"]
        ex = ""
        if d.any():
            a, b = j.loc[d, c+"_c"].iloc[0], j.loc[d, c+"_e"].iloc[0]
            ex = f"{a!r} -> {b!r}"
        rows.append({"column": c, "differs_as_stored": int(d.sum()), "example": ex})
    t = pd.DataFrame(rows); print(); print(t.to_string(index=False))
    differ = int(t["differs_as_stored"].sum())

    # unparseable values, reported on their own
    unp = {}
    for c in cols:
        num = pd.to_numeric(exp_s[c].str.strip().replace({"":None,"null":None}), errors="coerce")
        n = int(((~blank(exp_s[c])) & num.isna()).sum())
        if n: unp[c] = n

    # what pandas infers, which is what v2 actually measured
    ci = pd.read_csv(RUN/"training_candidate.csv", usecols=cols, low_memory=False)
    ei = pd.read_csv(exp_path, usecols=cols, low_memory=False)
    inf = pd.DataFrame({"column": cols,
                        "candidate_inferred": [str(ci[c].dtype) for c in cols],
                        "export_inferred":    [str(ei[c].dtype) for c in cols]})
    print(); print(inf.to_string(index=False))

    if unp:
        verdict("training", "candidate vs export (stored values)", "FAIL",
                "Values in the export do not parse as numbers:\n" +
                "\n".join(f"  {c}: {n:,}" for c, n in unp.items()), len(j), len(exp_s))
    elif differ:
        verdict("training", "candidate vs export (stored values)", "WARN",
                f"{differ:,} stored value(s) differ between the candidate partition and the "
                f"export. See the table above for the shape of the difference.",
                len(j), len(exp_s))
    else:
        verdict("training", "candidate vs export (stored values)", "PASS",
                f"Across {len(cols)} measurement columns and {len(j):,} joined rows, the "
                f"candidate partition and the export store byte-identical values. "
                f"v2's figure of 111,249 changed representations was an artifact of pandas "
                f"type inference in the check itself, not a change made by the pipeline. "
                f"That finding is withdrawn.", len(j), len(exp_s))

    same_inf = all(a == b for a, b in zip(inf["candidate_inferred"], inf["export_inferred"]))
    diagnostic("training", "pandas type inference on read",
               f"Read without an explicit dtype, pandas infers "
               f"{'the same type for both files' if same_inf else 'DIFFERENT types for the two files'}"
               f" in every measurement column. Inference turns a stored '5' into the float 5.0 in "
               f"memory, so anything that stringifies a parsed column will see '5.0'. That is a "
               f"property of the reader, and any consumer that needs the stored text must pass "
               f"dtype='string' and keep_default_na=False. It is not a pipeline change and no "
               f"documented rule governs it.")

## 5. Target distributions

Class shares should be close to unchanged. A material shift means rows were
removed unevenly across classes, which biases a model trained on the result.

In [ ]:
for tag, D in DATASETS.items():
    raw, cand = D["raw"], D["cand"]
    worst = ("n/a", 0.0)
    for c in TARGETS:
        if c not in raw.columns or c not in cand.columns: continue
        r = live(raw[c].str.strip()).value_counts(normalize=True)
        k = live(cand[c].str.strip()).value_counts(normalize=True)
        if r.empty or k.empty: continue
        idx = r.index.union(k.index)
        d = (k.reindex(idx).fillna(0) - r.reindex(idx).fillna(0)).abs().max()*100
        if d > worst[1]: worst = (c, d)
    if worst[1] == 0 and all(live(cand[c]).empty for c in TARGETS if c in cand.columns):
        verdict(tag, "target distributions", "BLOCKED",
                "This dataset carries no labels, so there is no distribution to compare.")
    elif worst[1] < 1:
        verdict(tag, "target distributions", "PASS",
                f"Largest class-share move {worst[1]:.2f}pp ({worst[0]}).", len(raw), len(raw))
    elif worst[1] < 5:
        verdict(tag, "target distributions", "WARN",
                f"{worst[0]} moved {worst[1]:.2f}pp. Check §6 for which class.", len(raw), len(raw))
    else:
        verdict(tag, "target distributions", "FAIL",
                f"{worst[0]} moved {worst[1]:.2f}pp - rows were removed unevenly across classes.",
                len(raw), len(raw))

## 6. Disappearing labels, traced to their rows

v2 accepted a disappearing label if it was whitespace-padded. **That is not a
justification.** A padded label can still be a real class, and "it had a stray
space" explains nothing about where its rows went.

This version requires, for every label present in raw and absent from the
cleaned candidate, that **each of its original rows** is one of exactly two
things:

1. **Quarantined**, under a reason code §2.1 verified against that row, or
2. **Relabelled**, with the rewrite recorded row by row in `label_changes.csv`
   *and* the destination being the replacement `docs/decisions.md` approves.

Anything else is a lost class and a FAIL. Whitespace is reported where it
occurs, but it never excuses a disappearance.

The approved replacements, from the label policies in `docs/decisions.md`:

| Target | From | To |
|---|---|---|
| Sub-Segment | `Cold / Flu` | `Cold/Flu` |
| Sub-Segment | `Other Lifestyle` | `Other Lifestyle CHC` |

In [ ]:
APPROVED_RELABELS = {
    ("Sub-Segment", "Cold / Flu"):      "Cold/Flu",
    ("Sub-Segment", "Other Lifestyle"): "Other Lifestyle CHC",
}

CH = {}
if len(LABEL_CHANGES):
    for _, r in LABEL_CHANGES.iterrows():
        CH.setdefault((r.get("dataset"), r["column"], str(r["old_value"])),
                      []).append((int(r["source_row"]), r["new_value"], r.get("rule","")))

for tag, D in DATASETS.items():
    raw, cand = D["raw"], D["cand"]
    kept = set(cand["source_row"])
    failures, padded_notes = [], []

    # A label changed on a RETAINED row must be recorded in label_changes.csv.
    # This is the strongest form of the check: it does not strip anything first,
    # so a value that lost only its padding still has to be accounted for.
    print(f"\n=== {tag}: label values changed on retained rows (exact comparison) ===")
    jj = cand.merge(raw[["source_row"]+[c for c in TARGETS if c in raw.columns]],
                    on="source_row", suffixes=("_c","_r"), how="inner")
    for c in TARGETS:
        if c+"_r" not in jj.columns: continue
        d = jj[jj[c+"_r"] != jj[c+"_c"]]
        rec = set(str(s) for s in
                  LABEL_CHANGES[(LABEL_CHANGES.get("dataset")==tag)
                                & (LABEL_CHANGES["column"]==c)]["source_row"]) \
              if len(LABEL_CHANGES) else set()
        unrec = d[~d["source_row"].astype(str).isin(rec)]
        print(f"  {c:16} {len(d):>4} changed, {len(d)-len(unrec):>3} recorded, "
              f"{len(unrec):>3} NOT recorded")
        for _, rr in unrec.head(5).iterrows():
            print(f"      source_row {rr['source_row']}: {rr[c+'_r']!r} -> {rr[c+'_c']!r}")
        if len(unrec):
            failures.append(f"{c}: {len(unrec)} retained row(s) had the label rewritten with "
                            f"no entry in label_changes.csv")

    for c in TARGETS:
        if c not in raw.columns or c not in cand.columns: continue
        rawvals = set(live(raw[c]))          # exact, not stripped
        candvals = set(live(cand[c]))        # exact, not stripped
        gone = sorted(rawvals - candvals)
        if not gone: continue
        print(f"\n--- {tag} / {c}: {len(gone)} label(s) in raw and not in the candidate")
        for lbl in gone:
            src = raw[raw[c] == lbl]
            srs = [int(s) for s in src["source_row"]]
            quar = [s for s in srs if str(s) not in kept]
            held = [s for s in srs if str(s) in kept]
            rec = CH.get((tag, c, lbl), [])
            rec_rows = {s for s, _, _ in rec}
            print(f"  {lbl[:62]!r}  {len(srs)} raw row(s)")
            if quar:
                print(f"      {len(quar)} quarantined (verified in 2.1): {quar[:8]}")
            accounted = set(quar)
            if held:
                dest = sorted({d for _, d, _ in rec})
                rules = sorted({u for _, _, u in rec if u})
                want = APPROVED_RELABELS.get((c, lbl.strip()))
                covered = [s for s in held if s in rec_rows]
                if covered:
                    ok = want is not None and dest == [want]
                    print(f"      {len(covered)} relabelled -> {dest} via {rules} "
                          f"[{'approved in decisions.md' if ok else 'NOT AN APPROVED REPLACEMENT'}]")
                    if ok: accounted |= set(covered)
                    else:  failures.append(f"{c}: {lbl!r} relabelled to {dest}, which "
                                           f"docs/decisions.md does not approve")
                missing = [s for s in held if s not in rec_rows]
                if missing:
                    print(f"      {len(missing)} row(s) RETAINED with no recorded relabel: {missing[:8]}")
                    failures.append(f"{c}: {lbl!r} - {len(missing)} row(s) retained but the "
                                    f"value is gone and no relabel is recorded")
            if lbl != lbl.strip():
                padded_notes.append(
                    f"{c}: {lbl[:44]!r} ({len(srs)} row(s)) carries stray whitespace. "
                    f"v2 would have excused its disappearance on that basis alone; here it "
                    f"was traced to its rows like every other value, and "
                    f"{len(quar)} of its {len(srs)} row(s) were quarantined under a reason "
                    f"verified in 2.1.")
            left = [s for s in srs if s not in accounted]
            if left and not any(f.startswith(f"{c}: {lbl!r}") for f in failures):
                failures.append(f"{c}: {lbl!r} - {len(left)} row(s) unaccounted for")

    shrunk = []
    for c in TARGETS:
        if c not in raw.columns or c not in cand.columns: continue
        r = live(raw[c].str.strip()).value_counts(); k = live(cand[c].str.strip()).value_counts()
        for lbl, n in k.items():
            if lbl in r.index and r[lbl] and 1 - n/r[lbl] > 0.20:
                shrunk.append((c, lbl, int(r[lbl]), int(n)))
    print(f"\n{tag}: surviving classes that lost more than 20% of their rows: {len(shrunk)}")
    for c, lbl, a, b in shrunk: print(f"    {c} {lbl!r}: {a} -> {b}")

    if failures:
        verdict(tag, "disappearing labels", "FAIL",
                "Labels whose rows are not accounted for:\n" +
                "\n".join("  "+f for f in failures), len(raw), len(raw))
    elif shrunk:
        verdict(tag, "disappearing labels", "WARN",
                f"Every disappearance is accounted for row by row, but {len(shrunk)} surviving "
                f"class(es) lost over 20% of their rows.", len(raw), len(raw))
    else:
        verdict(tag, "disappearing labels", "PASS",
                "Compared exactly, with nothing stripped first. Every label present in raw and "
                "absent from the candidate is accounted for row by row: its rows were either "
                "quarantined under a reason verified in 2.1, or relabelled to the replacement "
                "docs/decisions.md approves, with the rewrite recorded per row in "
                "label_changes.csv. Separately, every label value that differs on a RETAINED "
                "row has a matching entry in label_changes.csv, so no label was quietly "
                "rewritten. No disappearance is excused by whitespace alone.",
                len(raw), len(raw))

    if padded_notes:
        diagnostic(tag, "whitespace-padded values among the disappearing labels",
                   "\n".join(padded_notes))

## 7. Feature values on retained rows

v1 compared 76,219 of 83,938 rows and called it PASS. v2 fixed the coverage but
the transform test was still too permissive: it ran the path through
`dict.fromkeys`, which removes a repeated element **wherever it appears**. Under
that test `A > B > A > C` collapsing to `A > B > C` would have been reported as
explained, even though a level was genuinely lost.

**The strict rule, as it should have been from the start:**

| Raw | Candidate | Allowed? |
|---|---|---|
| `A > B > C` | `A>B>C` | yes — separator spacing only |
| `A > B > A > B` | `A > B` | yes — the **entire** path repeats exactly |
| `A > B > A > C` | `A > B > C` | **no** — this loses a level |
| anything else | | **no** |

A row whose change is not one of the first two is unexplained, and unexplained
changes are FAIL.

In [ ]:
APPROVED = {
    # column -> (what the transform does, where it is approved)
    "ProductCategory": ("separator spacing, plus collapse of a path that repeats in full",
                        "handoff README: 'safe spacing/category formatting'"),
}

def path_norm(s):
    """Separator spacing only. Nothing else is normalised away."""
    return re.sub(r"\s*>\s*", ">", str(s)).strip()

def whole_path_repeat(rawv, candv):
    """True only if raw is candidate repeated k>=2 times, exactly.
    'A>B>A>B' vs 'A>B' -> True.  'A>B>A>C' vs 'A>B>C' -> False."""
    rp, cp = path_norm(rawv).split(">"), path_norm(candv).split(">")
    if not cp or len(rp) <= len(cp) or len(rp) % len(cp): return False
    return rp == cp * (len(rp)//len(cp))

for tag, D in DATASETS.items():
    raw, cand = D["raw"], D["cand"]
    feats = [c for c in raw.columns
             if c in cand.columns and c not in TARGETS and c != "source_row"]
    j = (raw.set_index("source_row")[feats]
            .join(cand.set_index("source_row")[feats], lsuffix="_r", rsuffix="_c", how="inner"))
    print(f"--- {tag}: compared {len(j):,} rows of {len(cand):,} retained, "
          f"over {len(feats)} feature columns")
    if len(j) != len(cand):
        print(f"    ! {len(cand)-len(j):,} retained rows could not be joined")

    changed, unapproved = [], []
    for c in feats:
        d = j[c+"_r"] != j[c+"_c"]
        if not d.any(): continue
        changed.append((c, int(d.sum()), d.mean()))
        print(f"\n  {c}: {d.sum():,} rows changed ({d.mean():.2%})")
        for _, r in j.loc[d, [c+"_r", c+"_c"]].head(2).iterrows():
            print(f"      {str(r.iloc[0])[:62]!r}\n   -> {str(r.iloc[1])[:62]!r}")
        if c in APPROVED:
            what, where = APPROVED[c]
            print(f"      approved as: {what} ({where})")
            if c == "ProductCategory":
                sub = j.loc[d, [c+"_r", c+"_c"]]
                spacing = sub[c+"_r"].map(path_norm) == sub[c+"_c"].map(path_norm)
                repeat = pd.Series([whole_path_repeat(a, b) for a, b in
                                    zip(sub[c+"_r"], sub[c+"_c"])], index=sub.index)
                ok = spacing | repeat
                print(f"      separator spacing only          : {int(spacing.sum()):,}")
                print(f"      entire path repeated, collapsed : {int(repeat.sum()):,}")
                print(f"      UNEXPLAINED                     : {int((~ok).sum()):,}")
                for _, rr in sub[repeat].head(2).iterrows():
                    print(f"        full-repeat example:\n          {str(rr.iloc[0])[:96]}"
                          f"\n       -> {str(rr.iloc[1])[:96]}")
                for _, rr in sub[~ok].head(5).iterrows():
                    print(f"        UNEXPLAINED:\n          {str(rr.iloc[0])[:96]}"
                          f"\n       -> {str(rr.iloc[1])[:96]}")
                if not ok.all():
                    unapproved.append(f"{c}: {int((~ok).sum()):,} row(s) change beyond "
                                      f"separator spacing and a full-path repeat, so a path "
                                      f"level is lost")
        else:
            unapproved.append(f"{c}: {int(d.sum()):,} rows changed, no approving rule found")

    if unapproved:
        verdict(tag, "feature values on retained rows", "FAIL",
                "Changes with no approving rule:\n" + "\n".join("  "+u for u in unapproved),
                len(j), len(cand))
    elif changed:
        verdict(tag, "feature values on retained rows", "WARN",
                "Values changed in " + ", ".join(f"{c} ({n:,} rows)" for c,n,_ in changed) +
                ". Under the strict rule - separator spacing, or a path that repeats in full - "
                "every changed row is explained and no path level is lost. This is not a FAIL: "
                "policy 10 requires the full path to be preserved and it is. It is a WARN "
                "because the transform is documented only in the handoff README and not in "
                "docs/decisions.md, so it needs a recorded decision rather than a fix.",
                len(j), len(cand))
    else:
        verdict(tag, "feature values on retained rows", "PASS",
                "No feature value changed on any retained row.", len(j), len(cand))

## 8. Malformed labels

Whitespace-padded values in the target columns, raw against candidate.

**A surviving padded label is a diagnostic, not a FAIL.** `decisions.md` is
explicit: *"Target reasons 2 (leading-space/lowercase formatting) and 3 (rarity)
remain review-only"* and *"Case/spacing warnings and rare labels alone do not
quarantine"*. So a padded value that survives is the pipeline doing what the
policy says. It is only a FAIL if the policy said to remove it and it is still
there.

In [ ]:
for tag, D in DATASETS.items():
    raw, cand = D["raw"], D["cand"]
    rows = []
    for c in TARGETS:
        if c not in cand.columns: continue
        rb = int((live(raw[c]) != live(raw[c]).str.strip()).sum()) if c in raw.columns else 0
        cb = int((live(cand[c]) != live(cand[c]).str.strip()).sum())
        rows.append({"target": c, "raw": rb, "candidate": cb})
    t = pd.DataFrame(rows); print(f"--- {tag}"); print(t.to_string(index=False))
    left, found = int(t["candidate"].sum()), int(t["raw"].sum())

    verdict(tag, "malformed labels", "PASS",
            f"{found} padded value(s) in raw, {left} surviving in the candidate. Either "
            f"outcome complies: decisions.md keeps spacing warnings review-only, so a padded "
            f"label is not grounds for removal and its survival is not a breach. Note what "
            f"§6 shows about the mechanism - the pipeline does not strip padding, so a padded "
            f"value only leaves the data when its row is quarantined for an unrelated reason.",
            len(raw), len(raw))
    if left:
        diagnostic(tag, "padded label values surviving into the candidate",
                   f"{left} value(s) still carry leading or trailing whitespace and will be "
                   f"treated as distinct classes by anything that groups on the raw string. "
                   f"Review-only under decisions.md, so it is not a policy failure, but it "
                   f"is a real modelling hazard and belongs in feature engineering.")

## 9. Structural integrity of retained rows

Rewritten after review. The first version ran two cheap signals and concluded
there was no corruption, which the evidence did not support.

**Quoted records are included.** They were the ones most likely to be displaced,
and excluding them is how a displacement audit misses displacement.

**Each column is checked against its own domain** — numeric columns must hold
numbers, URL columns must hold URLs, the timestamp must not hold a URL. A value
in the wrong column is the actual signature of a shifted row.

**Shared vocabulary between targets is reported, not assumed to be corruption.**
A Platform called `Allergy` and a Segment called `Allergy` are different things
with the same name.

**An isolated bad value is a diagnostic, not a FAIL.** `decisions.md` policy 3
says a bad timestamp alone does nothing, and policy 5 says a bad URL alone does
not hold the row. So finding one of those among the retained rows is the pipeline
obeying policy. A FAIL here needs a value of the wrong kind in a column the
policy actually protects — a measurement column holding text, which is what
displacement looks like.

In [ ]:
for tag, D in DATASETS.items():
    raw, cand = D["raw"], D["cand"]
    kept_src = set(cand["source_row"])
    quoted_raw = int(raw["ProductName"].str.contains('"', na=False).sum())
    quoted_kept = int(cand["ProductName"].str.contains('"', na=False).sum())
    print(f"--- {tag}: {quoted_raw:,} raw rows contain a quote in ProductName; "
          f"{quoted_kept:,} of them are retained and included below")

    probs = {}
    for c in NUMERIC:
        if c in cand.columns:
            probs[f"{c} not numeric"] = int((~(cand[c].str.strip().str.match(r"^-?\d+(\.\d+)?$")
                                               | blank(cand[c]))).sum())
    for c in URLCOLS:
        if c in cand.columns:
            v = cand[c].str.strip().str.replace('"', '', regex=False).str.lower()
            probs[f"{c} not a URL"] = int(((~blank(cand[c])) & ~v.str.startswith(("http://","https://"))).sum())
    if "MDM_InsertDateTime" in cand.columns:
        ts = cand["MDM_InsertDateTime"].str.strip()
        probs["MDM_InsertDateTime holds a URL"] = int(ts.str.contains("http", case=False, na=False).sum())
        probs["MDM_InsertDateTime not numeric"] = int((~(ts.str.match(r"^\d+(\.\d+)?$")
                                                          | blank(cand["MDM_InsertDateTime"]))).sum())
    ragged = 0
    with open(DATASETS[tag]["run"].parent.parent/"provided"/
              ("Selfcare_Training_data (1).csv" if tag=="training" else "Selfcare_Target_data (1).csv"),
              encoding="utf-8", errors="ignore") as f:
        head = f.readline().count(",")
    for k, v in probs.items(): print(f"    {'OK     ' if v==0 else 'REVIEW '} {k:38s} {v}")

    vocab = {c: set(live(cand[c].str.strip())) for c in TARGETS if c in cand.columns}
    overlaps = []
    cols = list(vocab)
    for i, a in enumerate(cols):
        for b in cols[i+1:]:
            for val in sorted(vocab[a] & vocab[b]):
                overlaps.append((val, a, int((cand[a].str.strip()==val).sum()),
                                      b, int((cand[b].str.strip()==val).sum())))
    print(f"\n    values shared between two targets: {len(overlaps)}")
    for val, a, na, b, nb in overlaps:
        print(f"      {val!r}: {a} ({na:,}) and {b} ({nb:,})")

    # policy 3 and policy 5 make an isolated bad timestamp or URL review-only.
    REVIEW_ONLY = ("not a URL", "MDM_InsertDateTime not numeric")
    hard = {k: v for k, v in probs.items()
            if v and not any(k.endswith(s) or k == s for s in REVIEW_ONLY)}
    soft = {k: v for k, v in probs.items()
            if v and any(k.endswith(s) or k == s for s in REVIEW_ONLY)}

    if hard:
        verdict(tag, "structural integrity", "FAIL",
                "A measurement column holds a value of the wrong kind among RETAINED rows, "
                "which is the displacement signature the quarantine rules exist to catch:\n" +
                "\n".join(f"  {k}: {v:,}" for k, v in hard.items()), len(cand), len(cand))
    else:
        verdict(tag, "structural integrity", "PASS",
                f"No retained row holds a value of the wrong kind in a measurement column, "
                f"in a URL column, or a URL in the timestamp - checked over all "
                f"{len(cand):,} retained rows including the {quoted_kept:,} that contain a "
                f"quote. "
                + (f"{len(overlaps)} value(s) appear in two targets; they are distinct "
                   f"concepts sharing a name, not displacement." if overlaps else ""),
                len(cand), len(cand))

    if soft:
        diagnostic(tag, "review-only field issues among retained rows",
                   "\n".join(f"{k}: {v:,}" for k, v in soft.items()) +
                   "\nPolicy 3 (bad timestamp alone) and policy 5 (bad URL alone) make these "
                   "review-only, so their presence is compliance, not a breach. Listed so the "
                   "count is on the record.")
    if overlaps:
        diagnostic(tag, "vocabulary shared between two targets",
                   "\n".join(f"{v!r}: {a} ({na:,}) and {b} ({nb:,})"
                             for v, a, na, b, nb in overlaps) +
                   "\nDistinct concepts sharing a name. decisions.md forbids enforcing a "
                   "parent whitelist, so this is not actionable in cleaning.")

## 10. What left, and what it cost

§2.1 already established that every removal is supported by a documented rule
re-derived from the row itself. So the **policy** question is settled, and this
section is not a second verdict on it — it reports the **cost**, which is a
different thing and belongs in a diagnostic.

v1 called the rows with no product text "corrupted". That was wrong and stays
withdrawn: missing text is the condition rule 4 names, and a sparse record is
not a damaged one.

The labelled-row count is the part that matters to the project, and `Platform`
is broken out row by row because it is the scarcest of the six targets.

In [ ]:
for tag, D in DATASETS.items():
    raw, removed = D["raw"], D["removed"]
    if not removed:
        verdict(tag, "information loss", "PASS", "No rows removed."); continue
    gone = raw[raw["source_row"].isin(removed)]
    rdx = RD[RD["dataset"] == tag].set_index("source_row")
    gone = gone.join(rdx[["quarantine_reasons"]], on="source_row")

    notext = (blank(gone["ProductName"]) & blank(gone["ProductDescription"])
              & blank(gone["ProductContents"]))
    isnum = lambda s: s.str.strip().str.match(r"^-?\d+(\.\d+)?$") | blank(s)
    displaced = sum([(~isnum(gone[c])).astype(int) for c in NUMERIC]).astype(bool)
    urlts = gone["MDM_InsertDateTime"].str.contains("http", case=False, na=False)
    neither = int((~(displaced|urlts) & ~notext).sum())

    print(f"--- {tag}: {len(gone):,} removed")
    print(f"    a value sitting in the wrong column : {int((displaced|urlts).sum()):,}")
    print(f"    no product text at all              : {int(notext.sum()):,}")
    print(f"    neither of the above                : {neither:,}")

    tcols = [c for c in TARGETS if c in gone.columns]
    labelled = (~blank(gone[tcols])).any(axis=1)
    print(f"\n    removed rows carrying at least one label: {int(labelled.sum()):,} "
          f"of {len(gone):,}")
    per = {c: int((~blank(gone[c])).sum()) for c in tcols}
    for c, n in per.items():
        tot = int((~blank(raw[c])).sum())
        share = f"{n/tot*100:.2f}%" if tot else "n/a"
        print(f"      {c:16} {n:>5} removed of {tot:>7,} labelled in raw  ({share})")

    verdict(tag, "information loss", "PASS",
            f"All {len(gone):,} removals are supported - see 2.1, where each reason's "
            f"documented conditions were re-derived from the original row. "
            f"{int((displaced|urlts).sum()):,} rows hold a value in the wrong column; "
            f"{int(notext.sum()):,} have no product text, which is rule 4's condition and "
            f"not evidence of damage; {neither:,} fall outside both descriptions but are "
            f"still covered by a verified reason. The cost in labelled rows is recorded as "
            f"a diagnostic rather than folded into this verdict.", len(gone), len(gone))

    if int(labelled.sum()):
        lines = [f"{int(labelled.sum()):,} of the {len(gone):,} removed rows carried at least "
                 f"one label. This is not unlabelled junk being swept out."]
        for c, n in per.items():
            tot = int((~blank(raw[c])).sum())
            if n: lines.append(f"  {c}: {n} removed of {tot:,} labelled in raw "
                               f"({n/tot*100:.2f}%)" if tot else f"  {c}: {n} removed")
        if "Platform" in gone.columns and per.get("Platform"):
            lines.append("")
            lines.append("Platform is the scarcest target, so its rows are listed individually:")
            pl = gone[~blank(gone["Platform"])]
            for _, rr in pl.iterrows():
                codes = sorted({x.split(':',1)[1] for x in
                                str(rr["quarantine_reasons"]).split(';') if ':' in x})
                lines.append(f"  source_row {rr['source_row']:>6}  "
                             f"{str(rr['Platform'])[:30]:30}  {', '.join(codes)}")
            vc = pl["Platform"].str.strip().value_counts()
            tot = live(raw["Platform"].str.strip()).value_counts()
            lines.append("")
            lines.append("  per class:")
            for v, n in vc.items():
                lines.append(f"    {str(v)[:30]:30} {n:>3} removed of {int(tot.get(v,0)):>4} "
                             f"({n/int(tot.get(v,1))*100:.0f}% of that class)")
        diagnostic(tag, "labelled rows lost to quarantine", "\n".join(lines))

## 11. Summary

Verdicts and diagnostics are listed separately, because they answer different
questions. The verdicts answer *did the pipeline follow the rules*. The
diagnostics answer *what is still worth knowing about the data*.

In [ ]:
s = pd.DataFrame(VERDICTS)
if len(s):
    order = {"FAIL":0,"WARN":1,"BLOCKED":2,"PASS":3}
    s = s.sort_values(["status","dataset"],
                      key=lambda col: col.map(order) if col.name=="status" else col)
    print(s[["dataset","criterion","status","covered","population"]].to_string(index=False))
    print()
    c = s["status"].value_counts()
    print(f"FAIL {c.get('FAIL',0)}   WARN {c.get('WARN',0)}   "
          f"BLOCKED {c.get('BLOCKED',0)}   PASS {c.get('PASS',0)}"
          f"   (+ {len(DIAGNOSTICS)} diagnostics)")
    print()
    for _, r in s[s["status"] != "PASS"].iterrows():
        print(f"[{r['status']}] {r['dataset']} / {r['criterion']}")
        for line in str(r["detail"]).split("\n"): print("    " + line)
        print()

    print("="*78)
    print("DIAGNOSTICS - questionable values and costs that no documented rule prohibits.")
    print("None of these is a verdict on the pipeline.")
    print("="*78)
    for d in DIAGNOSTICS:
        print(f"\n[{d['dataset']}] {d['topic']}")
        for line in str(d["detail"]).split("\n"): print("    " + line)

    out = DATA/"interim"; out.mkdir(parents=True, exist_ok=True)
    s.to_csv(out/"cleaning_audit_summary.csv", index=False)
    pd.DataFrame(DIAGNOSTICS).to_csv(out/"cleaning_audit_diagnostics.csv", index=False)
    print("\nwritten to", out/"cleaning_audit_summary.csv")
    print("written to", out/"cleaning_audit_diagnostics.csv")

    under = s[(s["status"]=="PASS") & s["covered"].notna() & s["population"].notna()
              & (s["covered"] < s["population"])]
    print("\nPASS verdicts with incomplete coverage:", len(under), "(must be 0)")
    print("FAIL verdicts:", int((s["status"]=="FAIL").sum()))
else:
    print("no verdicts - run from the top")